In [8]:
import os
import pandas as pd
from pathlib import Path
import re

os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity\\OBIS')

# Fix dates

In [9]:
merged_OBIS = pd.read_parquet("merged_OBIS.parquet")

merged_OBIS["startDayOfYear"] = pd.to_datetime(
    merged_OBIS["date_start"], unit="ms", utc=True, errors="coerce"
).dt.dayofyear.astype("Int64")
merged_OBIS["endDayOfYear"] = pd.to_datetime(
    merged_OBIS["date_end"], unit="ms", utc=True, errors="coerce"
).dt.dayofyear.astype("Int64")

merged_OBIS.to_parquet("merged_OBIS.parquet", index=False)

# sampleSize

In [11]:
# get unique sampleSizeUnit values and save as CSV
merged_OBIS = pd.read_parquet("merged_OBIS.parquet")

out_path = Path("uniqueSampleSizeUnit.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

if "sampleSizeUnit" in merged_OBIS.columns:
    vals = pd.Series(merged_OBIS["sampleSizeUnit"].dropna().unique()).sort_values().reset_index(drop=True)
else:
    vals = pd.Series([], dtype="object")

vals.to_frame(name="sampleSizeUnit").to_csv(out_path, index=False)
print(f"Saved {out_path}")

Saved uniqueSampleSizeUnit.csv


In [12]:
# Extract trawling distance from sampleSizeValue
merged_OBIS["trawlingDistance_m"] = merged_OBIS["sampleSizeValue"].apply(
    lambda x: float(re.search(r"Distance\s+([\d.]+)\s+x\s+width", str(x)).group(1))
    if isinstance(x, str) and re.search(r"Distance\s+([\d.]+)\s+x\s+width", str(x))
    else pd.NA
).astype("Float64")

# Count entries added
count = merged_OBIS["trawlingDistance_m"].notna().sum()
print(f"Added {count:,} trawling distance entries to trawlingDistance_m column.")

# Save the updated dataframe
merged_OBIS.to_parquet("merged_OBIS.parquet", index=False)
print("Saved merged_OBIS.parquet")

Added 2,272 trawling distance entries to trawlingDistance_m column.
Saved merged_OBIS.parquet


In [13]:
# Extract area sampled from sampleSizeUnit and sampleSizeValue
merged_OBIS = pd.read_parquet("merged_OBIS.parquet")
merged_OBIS["areaSampled"] = pd.NA
merged_OBIS["areaSampledUnit"] = pd.NA

# Find rows where sampleSizeUnit is "1000 m2"
mask = merged_OBIS["sampleSizeUnit"] == "1000 m2"
merged_OBIS.loc[mask, "areaSampled"] = 1000.0
merged_OBIS.loc[mask, "areaSampledUnit"] = "m2"

# Convert to appropriate dtypes
merged_OBIS["areaSampled"] = merged_OBIS["areaSampled"].astype("Float64")
merged_OBIS["areaSampledUnit"] = merged_OBIS["areaSampledUnit"].astype("string")

# Count entries added
count = merged_OBIS["areaSampled"].notna().sum()
print(f"Added {count:,} area sampled entries to areaSampled column.")

# Save the updated dataframe
merged_OBIS.to_parquet("merged_OBIS.parquet", index=False)
print("Saved merged_OBIS.parquet")

Added 19,360 area sampled entries to areaSampled column.
Saved merged_OBIS.parquet


In [14]:
# Extract area sampled from sampleSizeUnit == "square metre"
mask_sq_metre = merged_OBIS["sampleSizeUnit"] == "square metre"
merged_OBIS.loc[mask_sq_metre, "areaSampled"] = pd.to_numeric(
    merged_OBIS.loc[mask_sq_metre, "sampleSizeValue"], errors="coerce"
).astype("Float64")
merged_OBIS.loc[mask_sq_metre, "areaSampledUnit"] = "m2"

# Count entries added
count_sq_metre = mask_sq_metre.sum()
print(f"Added {count_sq_metre:,} area sampled entries (square metre) to areaSampled and areaSampledUnit columns.")

# Save the updated dataframe
merged_OBIS.to_parquet("merged_OBIS.parquet", index=False)
print("Saved merged_OBIS.parquet")

Added 9,797 area sampled entries (square metre) to areaSampled and areaSampledUnit columns.
Saved merged_OBIS.parquet


In [15]:
# Add sampleSizeValue to individualCount when sampleSizeUnit == "Number of individuals"
vals = pd.to_numeric(merged_OBIS["sampleSizeValue"], errors="coerce")
mask_unit = merged_OBIS["sampleSizeUnit"] == "Number of individuals"
mask_apply = mask_unit & vals.notna()

# Ensure individualCount is numeric for arithmetic
merged_OBIS["individualCount"] = pd.to_numeric(merged_OBIS["individualCount"], errors="coerce")

# Add values
merged_OBIS.loc[mask_apply, "individualCount"] = (
    merged_OBIS.loc[mask_apply, "individualCount"].fillna(0) + vals.loc[mask_apply]
)

# Count and save
count = int(mask_apply.sum())
print(f"Added {count:,} entries to individualCount from sampleSizeValue.")

merged_OBIS.to_parquet("merged_OBIS.parquet", index=False)
print("Saved merged_OBIS.parquet")

Added 9,585 entries to individualCount from sampleSizeValue.
Saved merged_OBIS.parquet


In [16]:
# add volumeSampled_m3 where sampleSizeUnit indicates cubic metres
mask_volume = merged_OBIS["sampleSizeUnit"].astype("string").str.contains(
    r"(?i)\bm3\b|cubic[_\s-]*metre[s]?|cubic[_\s-]*meter[s]?", na=False
)

merged_OBIS["volumeSampled_m3"] = pd.NA
# reuse previously computed numeric vals if available
try:
    merged_OBIS.loc[mask_volume, "volumeSampled_m3"] = vals[mask_volume]
except NameError:
    merged_OBIS.loc[mask_volume, "volumeSampled_m3"] = pd.to_numeric(
        merged_OBIS.loc[mask_volume, "sampleSizeValue"], errors="coerce"
    )

merged_OBIS["volumeSampled_m3"] = merged_OBIS["volumeSampled_m3"].astype("Float64")

count = merged_OBIS["volumeSampled_m3"].notna().sum()
print(f"Added {count:,} volume entries to volumeSampled_m3 column.")

merged_OBIS.to_parquet("merged_OBIS.parquet", index=False)
print("Saved merged_OBIS.parquet")

Added 87,494 volume entries to volumeSampled_m3 column.
Saved merged_OBIS.parquet


# Check column coverage

In [17]:
column_coverage_OBIS = pd.DataFrame({
    "column": merged_OBIS.columns,
    "non_null_count": merged_OBIS.notna().sum().values,
    "total_rows": len(merged_OBIS),
})

column_coverage_OBIS["coverage_pct"] = (
    column_coverage_OBIS["non_null_count"] / column_coverage_OBIS["total_rows"] * 100
)

column_coverage_OBIS = column_coverage_OBIS.sort_values(
    by=["coverage_pct", "non_null_count"], ascending=False
)

column_coverage_OBIS.to_csv("column_coverage_OBIS.csv", index=False)

column_coverage_OBIS.head()

,column,non_null_count,total_rows,coverage_pct
0,basisOfRecord,1533558,1533558,100.0
10,decimalLatitude,1533558,1533558,100.0
11,decimalLongitude,1533558,1533558,100.0
27,kingdom,1533558,1533558,100.0
28,kingdomid,1533558,1533558,100.0


# Prepare for merging

In [18]:
merged_OBIS = pd.read_parquet("merged_OBIS.parquet")

dtype_spec = pd.read_excel(Path("..") / "column_data_types.xlsx", sheet_name=0)
dtype_map = dict(zip(dtype_spec["column"], dtype_spec["final"]))

column_map = {
    "sourceID": "id",
    "verbatimID": "occurrenceID",
    "datasetKey": "dataset_id",
    "datasetTitle": "datasetName",
    "decimalLatitudeStart": "decimalLatitude",
    "decimalLongitudeStart": "decimalLongitude",
    "footprintWKT": "footprintWKT",
    "coordinateUncertaintyInMeters": "coordinateUncertaintyInMeters",
    "depthA_m": "maximumDepthInMeters",
    "depthB_m": "minimumDepthInMeters",
    "eventID": "eventID",
    "parentEventID": "parentEventID",
    "waterBody": "waterBody",
    "locationID": "locationID",
    "verbatimDate": "eventDate",
    "startDayOfYear": "startDayOfYear",
    "endDayOfYear": "endDayOfYear",
    "year": "date_year",
    "eventRemarks": "eventRemarks",
    "verbatimScientificName": "originalScientificName",
    "acceptedScientificName": "scientificName",
    "identificationQualifier": "identificationQualifier",
    "lifeStage": "lifeStage",
    "sex": "sex",
    "taxonomicStatus": "taxonomicStatus",
    "taxonRank": "taxonRank",
    "phylum": "phylum",
    "class": "class",
    "order": "order",
    "family": "family",
    "genus": "genus",
    "specificEpithet": "specificEpithet",
    "validAuthority": "scientificNameAuthorship",
    "aphiaID": "aphiaID",
    "individualCount": "individualCount",
    "dynamicProperties": "dynamicProperties",
    "basisOfRecord": "basisOfRecord",
    "samplingEffort": "samplingEffort",
    "trawlingDistance_m": "trawlingDistance_m",
    "areaSampled": "areaSampled",
    "areaSampledUnit": "areaSampledUnit",
    "volumeSampled_m3": "volumeSampled_m3",
}

output_columns = [
    "ID",
    "sourceID",
    "verbatimID",
    "sourceDatabase",
    "datasetKey",
    "datasetTitle",
    "issue",
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "decimalLatitudeStop",
    "decimalLongitudeStop",
    "footprintWKT",
    "coordinateUncertaintyInMeters",
    "hasGeospatialIssues",
    "depthA_m",
    "depthAType",
    "depthB_m",
    "depthBType",
    "gebcoDepth",
    "eventID",
    "parentEventID",
    "waterBody",
    "largeMarineEcosystem",
    "locationID",
    "verbatimDate",
    "startDayOfYear",
    "endDayOfYear",
    "year",
    "eventRemarks",
    "verbatimScientificName",
    "acceptedScientificName",
    "identificationQualifier",
    "lifeStage",
    "sex",
    "taxonomicStatus",
    "taxonomicIssue",
    "taxonRank",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "specificEpithet",
    "validAuthority",
    "aphiaID",
    "individualCount",
    "areaSampled",
    "areaSampledUnit",
    "volumeSampled_m3",
    "samplingEffort",
    "trawlingDistance_m",
    "biomass",
    "biomassUnit",
    "dynamicProperties",
    "basisOfRecord",
    "occurrenceRemarks",
    "gear",
    "gearCategory",
    "meshSize_mm",
    "sieveFraction_µm",
    "flagged",
    "flaggedComment", 
    "acceptedTaxonKey"
]


def normalize_dtype(dtype_name):
    dtype_name = str(dtype_name).strip().lower()
    return {
        "int32": "Int32",
        "float32": "Float32",
        "float64": "Float64",
        "boolean": "boolean",
        "string": "string",
    }.get(dtype_name, dtype_name)


def empty_series(index, dtype_name):
    dtype_name = normalize_dtype(dtype_name)
    if dtype_name in {"Int32", "Float32", "Float64", "boolean", "string"}:
        return pd.Series(pd.NA, index=index, dtype=dtype_name)
    return pd.Series(pd.NA, index=index, dtype="string")


def coerce_series(series, dtype_name):
    dtype_name = normalize_dtype(dtype_name)
    if dtype_name == "boolean":
        return series.astype("boolean")
    if dtype_name in {"Int32", "Float32", "Float64"}:
        return pd.to_numeric(series, errors="coerce").astype(dtype_name)
    return series.astype("string")


obis_final = pd.DataFrame(index=merged_OBIS.index)
missing_sources = []

for col in output_columns:
    target_dtype = dtype_map.get(col, "string")
    if col in column_map:
        src = column_map[col]
        if src in merged_OBIS.columns:
            obis_final[col] = coerce_series(merged_OBIS[src], target_dtype)
        else:
            obis_final[col] = empty_series(merged_OBIS.index, target_dtype)
            missing_sources.append(src)
    else:
        obis_final[col] = empty_series(merged_OBIS.index, target_dtype)

obis_final = obis_final[output_columns]
obis_final.to_parquet("obis_final.parquet", index=False)

print(f"Saved obis_final.parquet with {len(obis_final):,} rows and {len(obis_final.columns)} columns.")
if missing_sources:
    print("Missing source columns in merged_OBIS:", sorted(set(missing_sources)))

Saved obis_final.parquet with 1,533,558 rows and 63 columns.


In [19]:
obis_final = pd.read_parquet("obis_final.parquet")
obis_final["sourceDatabase"] = "OBIS"
obis_final.to_parquet("obis_final.parquet", index=False)
print("Added sourceDatabase column with value 'OBIS'. Saved obis_final.parquet.")

Added sourceDatabase column with value 'OBIS'. Saved obis_final.parquet.


In [20]:
obis_final.loc[obis_final["depthA_m"].notna(), "depthAType"] = "maximum"
obis_final.loc[obis_final["depthB_m"].notna(), "depthBType"] = "minimum"

obis_final.to_parquet("obis_final.parquet", index=False)
obis_final[["depthA_m", "depthAType", "depthB_m", "depthBType"]].head()
obis_final.to_parquet("obis_final.parquet", index=False)
print("Saved obis_final.parquet.")

Saved obis_final.parquet.


# Check data types

In [21]:
obis_final = pd.read_parquet("obis_final.parquet")

# Column type overview
column_types = pd.DataFrame({
    "column": obis_final.columns,
    "dtype": obis_final.dtypes.astype(str).values,
    "non_null_count": obis_final.notna().sum().values,
    "null_count": obis_final.isna().sum().values,
    "non_null_pct": (obis_final.notna().sum().values / len(obis_final)) * 100,
})

column_types = column_types.sort_values(
    by=["non_null_count", "column"], ascending=[False, True]
 )

column_types.to_csv("obis_final_column_types.csv", index=False)

column_types.head(20)

,column,dtype,non_null_count,null_count,non_null_pct
30,acceptedScientificName,string,1533558,0,100.000000
44,aphiaID,string,1533558,0,100.000000
54,basisOfRecord,string,1533558,0,100.000000
4,datasetKey,string,1533558,0,100.000000
7,decimalLatitudeStart,Float64,1533558,0,100.000000
8,decimalLongitudeStart,Float64,1533558,0,100.000000
37,phylum,string,1533558,0,100.000000
3,sourceDatabase,str,1533558,0,100.000000
1,sourceID,string,1533558,0,100.000000
29,verbatimScientificName,string,1532890,668,99.956441
